# Phase 4 — Decentralized Baseline Inventory Policy

**Phase:** 4 of 8 (see `PROJECT_CHARTER.md`)  
**Purpose:** build the baseline that Phase 5's multi-echelon optimizer must beat.
Without this, any later "X% cost reduction" claim has no denominator.

This is a **decentralized** `(s, S)` policy: every store-SKU computes its own
reorder point and order-up-to level *independently*, ignoring the rest of the
network. That lack of coordination is deliberate — it's the realistic
each-node-optimizes-alone practice that the multi-echelon optimizer improves on.

### Tooling note (read before citing SimPy)
SimPy was unavailable in the build environment (no network access). The
discrete-event simulation in `src/optimization/des_engine.py` implements the
identical event-loop behaviour directly in NumPy so it runs and is verified.
Describe it in your methodology as *"a discrete-event inventory simulation;
SimPy was unavailable, so the event loop is implemented directly."*


## 0. Setup + config


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from src.models.router import route_skus
from src.models.seasonal_model import FourierSeasonalForecaster
from src.models.gbm_model import GBMForecaster
from src.models.croston import CrostonOrBaseline
from src.optimization.policy_math import service_z, safety_stock, eoq, reorder_point, order_up_to
from src.optimization.des_engine import simulate_sS

pd.set_option('display.width', 130)
plt.rcParams['figure.dpi'] = 100
ROOT = Path.cwd().parent

demand = pd.read_csv(ROOT / 'data/raw/demand.csv', parse_dates=['date'])
skus = pd.read_csv(ROOT / 'data/raw/sku_master.csv')
br = yaml.safe_load(open(ROOT / 'configs/business_rules.yaml'))
mp = yaml.safe_load(open(ROOT / 'configs/model_params.yaml'))

costs = br['costs']
floors = br['service_floors']
LT_STORE = br['assumptions']['lead_time_days']['dc_to_store']
MOQ, PACK = br['constraints']['min_order_qty'], br['constraints']['pack_size']
HOLDOUT = int(mp['simulation']['holdout_days'])
stores = {s['id']: s for s in br['network']['stores']}
print(f'lead time (DC->store): {LT_STORE}d | holdout: {HOLDOUT}d | floors: {floors}')


## 1. Safety stock from FORECAST RESIDUALS (the Phase 3 -> Phase 4 handoff)

Safety stock uses `SS = z * sigma_d * sqrt(L)`. The key modelling choice:
`sigma_d` is the standard deviation of the **forecast error** (actual minus
predicted on the holdout), not raw demand variance. This correctly credits
well-forecast SKUs with needing a smaller buffer — a SKU we predict accurately
doesn't need as much protection as one we predict poorly, even if their raw
demand variance is identical. This is the concrete reason Phase 3 had to come
before Phase 4.


In [ ]:
routing, _ = route_skus(skus)
sku_info = skus.set_index('sku_id')

net_daily = (demand.groupby(['sku_id', 'date'], observed=True)
             .agg(units=('units', 'sum'), on_promo=('on_promo', 'max'))
             .reset_index().sort_values(['sku_id', 'date']))
full_idx = pd.date_range(net_daily.date.min(), net_daily.date.max(), freq='D')
cutoff = net_daily['date'].max() - pd.Timedelta(days=HOLDOUT - 1)

def forecast_residual_std(sku_id):
    sub = net_daily[net_daily.sku_id == sku_id].set_index('date').reindex(full_idx)
    sub['units'] = sub['units'].fillna(0); sub['on_promo'] = sub['on_promo'].fillna(False)
    sub = sub.reset_index().rename(columns={'index': 'date'})
    tr, te = sub[sub.date < cutoff], sub[sub.date >= cutoff]
    model = routing.set_index('sku_id').loc[sku_id, 'model']
    yv = te['units'].to_numpy()
    if model == 'fourier_seasonal':
        m = FourierSeasonalForecaster().fit(tr['date'], tr['units'].to_numpy(), tr['on_promo'].to_numpy())
        pred = m.predict(te['date'], te['on_promo'].to_numpy())
    elif model == 'gbm':
        m = GBMForecaster().fit(tr[['date', 'units', 'on_promo']])
        pred = m.predict(te['date'], te['on_promo'].to_numpy())
    else:
        m = CrostonOrBaseline().fit(tr['units'].to_numpy()); pred = m.predict(len(yv))
    return float(np.std(yv - pred))

resid_std = {sid: forecast_residual_std(sid) for sid in routing.sku_id}
net_mean = {sid: net_daily[net_daily.sku_id == sid]['units'].mean() for sid in routing.sku_id}
print('Residual std computed for', len(resid_std), 'SKUs.')


## 2. Build the per-store-SKU (s, S) policy and simulate it

For each of the 3 stores x 45 SKUs = 135 policies: compute EOQ, residual-based
safety stock (the network residual apportioned to the store by its demand
share), reorder point and order-up-to level, then run the discrete-event
simulation over the 90-day holdout to get real costs and fill rate.


In [ ]:
def store_series(store_id, sku_id):
    sub = (demand[(demand.store_id == store_id) & (demand.sku_id == sku_id)]
           .groupby('date')['units'].sum().reindex(full_idx).fillna(0))
    return sub.to_numpy()

results = []
for store_id, store in stores.items():
    for sid in routing.sku_id:
        abc = sku_info.loc[sid, 'abc_class']
        unit_cost = float(sku_info.loc[sid, 'unit_cost'])
        z = service_z(floors[abc])

        series = store_series(store_id, sid)
        n_train = (full_idx < cutoff).sum()
        train_d, sim_d = series[:n_train], series[n_train:]
        mean_daily = float(train_d.mean())

        store_share = mean_daily / max(net_mean[sid], 1e-9)
        sigma_err = resid_std[sid] * store_share

        h = unit_cost * costs['holding_rate_annual']
        ss = safety_stock(z, sigma_err, LT_STORE)
        Q = eoq(mean_daily * 365.0, costs['ordering_cost'], h)
        s = reorder_point(mean_daily, LT_STORE, ss)
        S = order_up_to(s, Q)

        r = simulate_sS(sim_d, s, S, LT_STORE, unit_cost,
                        costs['holding_rate_annual'], costs['ordering_cost'],
                        costs['shortage_penalty_per_unit'], MOQ, PACK)
        results.append({
            'store_id': store_id, 'sku_id': sid, 'abc_class': abc,
            'safety_stock': ss, 'reorder_pt': s, 'order_up_to': S, 'EOQ': Q,
            'holding_cost': r.total_holding_cost, 'ordering_cost': r.total_ordering_cost,
            'shortage_cost': r.total_shortage_cost, 'total_cost': r.total_cost,
            'fill_rate': r.fill_rate, 'avg_on_hand': r.avg_on_hand,
        })
R = pd.DataFrame(results)
print(f'{len(R)} store-SKU policies simulated over the {HOLDOUT}-day holdout.')


## 3. Baseline results: total cost and realized service


In [ ]:
print(f"Total network cost (90d): ${R['total_cost'].sum():,.0f}")
print(f"  Holding:  ${R['holding_cost'].sum():,.0f}")
print(f"  Ordering: ${R['ordering_cost'].sum():,.0f}")
print(f"  Shortage: ${R['shortage_cost'].sum():,.0f}")

print('\nRealized fill rate vs floor, by ABC class:')
for abc in ['A', 'B', 'C']:
    sub = R[R.abc_class == abc]
    fr = sub['fill_rate'].mean()
    flag = 'OK' if fr >= floors[abc] else 'BELOW FLOOR'
    print(f'  {abc}: {fr:.1%} vs floor {floors[abc]:.0%}  [{flag}]  slack {fr-floors[abc]:+.1%}')


## 4. A real finding: the baseline OVER-SERVES, and it's an EOQ batch effect

The baseline hits ~99.7–99.9% fill on every class — well above the 98/95/90%
floors. This is not a bug, and importantly it is **not** caused by oversized
safety stock. The diagnostic below shows safety stock is tiny for C-items;
the excess inventory comes from **EOQ ordering large batches relative to
demand** for low-volume items (EOQ scales with the $75 ordering cost, which
is large relative to a cheap item's holding cost).

**Why this matters for Phase 5:** it separates two distinct sources of the
cost reduction the optimizer will achieve —
1. *service-floor right-sizing* (letting cheap C-items fall toward their 90%
   floor instead of incidental 99.9%), and
2. *multi-echelon coordination* (risk-pooling safety stock at the DC).

These are different claims. Reporting the baseline's over-service honestly now
means Phase 5 can attribute its win to the right cause, rather than letting a
'free' batch-size effect masquerade as a coordination benefit.


In [ ]:
diag = R.groupby('abc_class').agg(
    holding_cost=('holding_cost', 'sum'),
    mean_fill=('fill_rate', 'mean'),
    mean_safety_stock=('safety_stock', 'mean'),
    mean_avg_on_hand=('avg_on_hand', 'mean'),
    mean_EOQ=('EOQ', 'mean'),
).reindex(['A', 'B', 'C'])
diag['holding_pct_of_total'] = diag['holding_cost'] / diag['holding_cost'].sum()
diag.round(2)


Read the `mean_safety_stock` vs `mean_avg_on_hand` columns for C-items: safety
stock is ~2 units but average on-hand is ~140 — the gap is pure EOQ batch size,
confirming the over-service is a batching artifact, not over-buffering. Note
also that A-items carry ~60% of holding cost with the *least* service slack —
so that's where genuine multi-echelon optimization (not batch-trimming) will
have to do the real work in Phase 5.


## 5. Visualize: cost composition and an example inventory trajectory


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: holding cost by ABC class
hc = R.groupby('abc_class')['holding_cost'].sum().reindex(['A', 'B', 'C'])
axes[0].bar(hc.index, hc.values, color=['#1D9E75', '#5B8FF9', '#D85A30'])
axes[0].set_title('Baseline holding cost by ABC class (90d)', fontsize=10, loc='left')
axes[0].set_ylabel('holding cost ($)')

# Right: one example on-hand trajectory (an A-item at the flagship store)
ex_sku = R[R.abc_class == 'A'].iloc[0]['sku_id']
series = store_series('S1', ex_sku)[(full_idx >= cutoff)]
n_train = (full_idx < cutoff).sum()
row = R[(R.store_id == 'S1') & (R.sku_id == ex_sku)].iloc[0]
r = simulate_sS(series, row['reorder_pt'], row['order_up_to'], LT_STORE,
                float(sku_info.loc[ex_sku, 'unit_cost']), costs['holding_rate_annual'],
                costs['ordering_cost'], costs['shortage_penalty_per_unit'], MOQ, PACK)
axes[1].plot(r.daily_on_hand, color='#1D9E75', lw=1.0)
axes[1].axhline(row['reorder_pt'], color='#D85A30', ls='--', lw=0.8, label='reorder point s')
axes[1].axhline(row['order_up_to'], color='#888780', ls=':', lw=0.8, label='order-up-to S')
axes[1].set_title(f'On-hand trajectory: {ex_sku} @ S1', fontsize=10, loc='left')
axes[1].set_xlabel('day of holdout'); axes[1].set_ylabel('units on hand')
axes[1].legend(fontsize=8, frameon=False)
fig.tight_layout(); plt.show()


## 6. Persist baseline results for Phase 5 comparison


In [ ]:
out = ROOT / 'data' / 'processed'
out.mkdir(parents=True, exist_ok=True)
R.to_csv(out / 'baseline_policy_results.csv', index=False)

summary = {
    'total_cost': float(R['total_cost'].sum()),
    'holding_cost': float(R['holding_cost'].sum()),
    'ordering_cost': float(R['ordering_cost'].sum()),
    'shortage_cost': float(R['shortage_cost'].sum()),
    'fill_rate_by_abc': {abc: float(R[R.abc_class == abc]['fill_rate'].mean()) for abc in ['A','B','C']},
    'holdout_days': HOLDOUT,
}
import json
with open(out / 'baseline_summary.json', 'w') as fh:
    json.dump(summary, fh, indent=2)
print('Saved baseline_policy_results.csv and baseline_summary.json to data/processed/')
print(json.dumps(summary, indent=2))


## 7. Summary

| Item | Result |
|---|---|
| Policy type | Decentralized (s, S), per store-SKU, independent of network |
| Safety stock basis | Forecast residual std (Phase 3 handoff), not raw demand |
| Total network cost (90d) | See Section 3 |
| Service vs floors | All classes ABOVE floor — baseline over-serves |
| Over-service cause | EOQ batch size on low-volume items, NOT safety stock (diagnosed, Section 4) |
| Saved for Phase 5 | `data/processed/baseline_policy_results.csv`, `baseline_summary.json` |

**Next: Phase 5 — Multi-Echelon Optimization (critical path).** Build the
coordinated policy that pools safety stock at the DC and right-sizes orders,
then compare its cost to this baseline *at equal-or-better service floors*.
Attribute the win explicitly to coordination vs. batch/floor right-sizing,
per the Section 4 finding.
